# 🏥 MedSafe Copilot (P-054) — Kaggle GPU Embedding Pipeline (BAAI/bge-m3 -> Qdrant Cloud)

Notebook này dùng để chạy trên **Kaggle (GPU T4)** với tính năng **RESUME AN TOÀN 100%**:
1. Cài đặt các thư viện `sentence-transformers`, `qdrant-client`, `torch` (CUDA acceleration).
2. Nạp toàn bộ 772 file JSON bóc tách chi tiết từ thư mục Kaggle Dataset (`extracted_leaflets`).
3. Sử dụng **UUIDv5 cố định theo File + Vị trí Chunk**, giúp Qdrant tự động **bỏ qua / ghi đè thông minh** nếu bạn bấm Stop hay chạy lại nửa chừng (không bao giờ bị nhân bản trùng lặp vector).
4. Encode bằng mô hình **`BAAI/bge-m3`** (1024-dim, cosine) trên GPU T4 và push trực tiếp lên Qdrant Cloud.

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q sentence-transformers qdrant-client tqdm torch

import os
import json
import glob
import uuid
from pathlib import Path
import torch
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance, PointStruct, Filter,
    FieldCondition, MatchValue, MatchAny, PayloadSchemaType
)

# Kiểm tra thiết bị GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Đang sử dụng thiết bị: {device}")
if device == "cuda":
    print(f"   GPU Device: {torch.cuda.get_device_name(0)}")

## 🔑 2. Cấu hình Thông số Kết nối Qdrant Cloud & Đường dẫn Dữ liệu

In [ ]:
# Thông số kết nối Qdrant Cloud (Điền thông tin của bạn tại đây)
QDRANT_URL = "https://YOUR_CLUSTER_ID.us-east-1-0.aws.cloud.qdrant.io:6333"
QDRANT_API_KEY = "YOUR_QDRANT_CLOUD_API_KEY"
COLLECTION_NAME = "hdsd_medsafe_chunks"

# Đường dẫn thư mục Kaggle Dataset chứa 772 file JSON (ví dụ: '/kaggle/input/p054-leaflets-json/extracted_leaflets')
DATA_DIR = "/kaggle/input/p054-leaflets-json/extracted_leaflets"

print("✅ Cấu hình hoàn tất!")

## 🧠 3. Tải Mô hình BAAI/bge-m3 & Khởi tạo Collection trên Qdrant Cloud

In [ ]:
# 1. Load Mô hình Embedding BAAI/bge-m3
print("⏳ Đang tải mô hình BAAI/bge-m3 trên GPU...")
embedder = SentenceTransformer("BAAI/bge-m3", device=device)
embedder.max_seq_length = 8192  # BGE-M3 hỗ trợ cửa sổ context 8192 tokens
print(f"✅ Đã nạp mô hình BAAI/bge-m3 thành công! (Dimension = {embedder.get_sentence_embedding_dimension()})")

# 2. Khởi tạo Qdrant Client
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)

# 3. Tạo Collection nếu chưa tồn tại (Vector size = 1024, Cosine Distance)
collections = [c.name for c in qdrant.get_collections().collections]
if COLLECTION_NAME not in collections:
    print(f"--> Đang tạo Collection '{COLLECTION_NAME}' trên Qdrant Cloud...")
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )
    print("✅ Đã tạo thành công Collection mới!")
else:
    print(f"ℹ️ Collection '{COLLECTION_NAME}' đã tồn tại sẵn trên Qdrant Cloud.")

# 4. Tạo các Payload Index để Lọc Nhanh & Chính Xác 100% không bị lẫn thuốc
print("--> Khởi tạo Payload Indexes cho Keyword Search...")
for field in ["canonical_ingredients", "brand_name", "file_name", "section_name"]:
    try:
        qdrant.create_payload_index(
            collection_name=COLLECTION_NAME,
            field_name=field,
            field_schema=PayloadSchemaType.KEYWORD,
        )
    except Exception:
        pass
print("✅ Đã thiết lập thành công Payload Indexes!")

## 📦 4. Đọc Dữ Liệu Các File JSON từ Kaggle Dataset

In [ ]:
def load_all_drug_files(data_dir: str) -> list:
    drugs = []
    p = Path(data_dir)
    if not p.exists():
        matches = list(Path("/kaggle/input").rglob("extracted_leaflets"))
        if matches:
            p = matches[0]
        else:
            p = Path("data/extracted_leaflets")
            
    json_files = list(p.glob("*.json"))
    print(f"--> Tìm thấy {len(json_files)} file JSON trong thư mục {p}.")
    for f in json_files:
        try:
            with open(f, "r", encoding="utf-8") as jf:
                drugs.append(json.load(jf))
        except Exception as e:
            print(f"❌ Lỗi đọc {f.name}: {e}")
    return drugs

drug_records = load_all_drug_files(DATA_DIR)
print(f"✅ Đã nạp thành công {len(drug_records)} thuốc sẵn sàng bóc tách Chunks.")

## 🚀 5. Chạy Embedding BGE-M3 (Cơ chế Resume / Idempotent UUID) & Upsert Qdrant

In [ ]:
all_chunks_text = []
all_payloads = []
all_point_ids = []

for drug in drug_records:
    file_name = drug.get("file_name", "")
    brand_name = drug.get("brand_name", "")
    raw_ingredient = drug.get("active_ingredient", "")
    
    canonical_ingredients = [brand_name.lower()]
    if raw_ingredient:
        canonical_ingredients.append(raw_ingredient.lower())

    chunks = drug.get("chunks", [])
    for idx, c in enumerate(chunks):
        chunk_text = c.get("text", "").strip()
        if not chunk_text:
            continue
        
        section_name = c.get("section", "CHUNG") or "CHUNG"
        rich_text_to_embed = f"Thuốc: {brand_name} | Mục: {section_name}\n{chunk_text}"
        
        payload = {
            "file_name": file_name,
            "brand_name": brand_name,
            "canonical_ingredients": canonical_ingredients,
            "section_name": section_name,
            "chunk_index": idx,
            "char_start": c.get("char_start", 0),
            "char_end": c.get("char_end", 0),
            "text": chunk_text
        }
        
        # ID định danh cố định theo UUIDv5 -> Đảm bảo Idempotent & Resume an toàn
        point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{file_name}_{idx}_{c.get('char_start', 0)}"))
        
        all_chunks_text.append(rich_text_to_embed)
        all_payloads.append(payload)
        all_point_ids.append(point_id)

print(f"📊 TỔNG SỐ CHUNKS CẦN EMBED & UPSERT: {len(all_chunks_text)} chunks.")

BATCH_SIZE = 64
print(f"--> Đang tính toán Vector Embeddings với batch_size={BATCH_SIZE} trên GPU...")

for i in tqdm(range(0, len(all_chunks_text), BATCH_SIZE), desc="Embedding & Upserting (Resume Safe)"):
    batch_texts = all_chunks_text[i : i + BATCH_SIZE]
    batch_payloads = all_payloads[i : i + BATCH_SIZE]
    batch_ids = all_point_ids[i : i + BATCH_SIZE]
    
    embeddings = embedder.encode(batch_texts, batch_size=len(batch_texts), normalize_embeddings=True).tolist()
    
    points = [
        PointStruct(id=p_id, vector=emb, payload=p_load)
        for p_id, emb, p_load in zip(batch_ids, embeddings, batch_payloads)
    ]
    
    # Upsert vào Qdrant Cloud (tự ghi đè/bỏ qua trùng lặp nhờ UUIDv5 cố định)
    qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

print(f"🎉 NẠP THÀNH CÔNG HOÀN TOÀN {len(all_chunks_text)} VECTORS LÊN QDRANT CLOUD!")

## 🔍 6. Thử Nghiệm Tra Cứu RAG Chuẩn (100% Không Lẫn Thuốc Khác)

In [ ]:
def query_drug_rag(query_text: str, target_brand_name: str = None, top_k: int = 3):
    query_vector = embedder.encode(query_text, normalize_embeddings=True).tolist()
    
    query_filter = None
    if target_brand_name:
        query_filter = Filter(
            must=[
                FieldCondition(
                    key="brand_name",
                    match=MatchValue(value=target_brand_name)
                )
            ]
        )
    
    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        query_filter=query_filter,
        limit=top_k
    )
    
    print(f"=== KẾT QUẢ TRA CỨU: '{query_text}' (Filter: {target_brand_name}) ===")
    for r in results:
        print(f"[Score: {r.score:.4f}] Thuốc: {r.payload['brand_name']} | Mục: {r.payload['section_name']}")
        print(f"    Đoạn trích: {r.payload['text'][:150]}...\n")

query_drug_rag("Tác dụng phụ gây đau bụng đầy hơi", top_k=2)